# NumPy and pandas

## Learning objectives

By the end of this notebook you will be able to:

- create and inspect a NumPy `ndarray` and use vectorised arithmetic on it;
- select elements with boolean masks and standardise a column;
- load a `DataFrame`, inspect its shape, types, and missing values;
- select, filter, and derive columns with pandas rather than loops;
- summarise with `groupby` and combine tables with `merge`;
- reshape a summary with `pivot_table`.

## Concept

Python lists are flexible but slow for arithmetic. **NumPy** provides fixed-type, contiguous
arrays with vectorised operations, and **pandas** builds labelled tables on top of them. Together
they are the workhorses of the rest of this repository, so this notebook moves from a raw array
to a grouped and merged table.

A NumPy **ndarray** is a block of values that all share one dtype. Because the block is
contiguous and typed, `array * 2 + 1` runs in compiled code over the whole block instead of
executing a Python loop per element. The same idea gives boolean masks: `array > 4000`
produces an array of `True` / `False`, which can index, count, or average.

A pandas **DataFrame** is a dict of typed columns that share a row index. `loc` selects by
label, `iloc` by position, a boolean Series filters rows, `.assign` creates a new column, and
`.groupby(...).agg(...)` computes per-group summaries. `merge` joins two frames on a shared
column, and `pivot_table` reshapes a long summary into a grid.

The mental shift is from "loop over rows and update a result" to "describe the operation on
whole columns". Vectorised code is shorter, faster, and less bug-prone once you trust it.

## Worked example

### Set up the plot theme and load the table

The shared `set_theme` helper keeps every figure in the repository consistent.

In [ ]:
import numpy as np
import pandas as pd

from ds_practice import load_penguins, set_theme, histogram, scatterplot

set_theme()
penguins = load_penguins()
print("shape:", penguins.shape)
penguins.head()

### From a column to an ndarray

`Series.to_numpy()` gives a plain array. From there we can compute summary statistics, apply a
formula to every element at once, and select with a mask — no Python loop.

In [ ]:
mass = penguins["body_mass_g"].dropna().to_numpy()
print("dtype:", mass.dtype, "| shape:", mass.shape)
print("mean:", mass.mean().round(1), "g")
print("std :", mass.std().round(1), "g")
print("min / max:", mass.min(), "/", mass.max())

standardised = (mass - mass.mean()) / mass.std()
print("\nfirst five standardised scores:", standardised[:5].round(2))

over_four_kg = mass[mass > 4000]
print("over 4 kg:", over_four_kg.size, "penguins",
      f"({over_four_kg.size / mass.size:.1%} of the clean rows)")

### A first DataFrame view

`info` reports dtypes and non-null counts, which is the quickest way to spot a column with
missing values.

In [ ]:
penguins.info()
print()
print("missing per column:")
print(penguins.isna().sum())

### Selecting, filtering, and deriving

We pick columns by name, filter with a boolean Series, and add a kilogram column with `.assign`.
The final line groups by species and asks for several summaries at once.

In [ ]:
numeric = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
print(penguins[["species", "island", *numeric]].head(3).to_string(index=False))

gentoo = penguins[penguins["species"] == "Gentoo"]
print("\nGentoo rows:", len(gentoo))

penguins = penguins.assign(mass_kg=penguins["body_mass_g"] / 1000)
by_species = penguins.groupby("species")["mass_kg"].agg(["count", "mean", "std", "min", "max"])
print("\nmass (kg) by species:")
print(by_species.round(3))

### Merging a lookup table

A merge adds columns from a second frame that share a key. Here we label each island with a
region, then summarise the (now larger) frame by that new column.

In [ ]:
island_region = pd.DataFrame(
    {
        "island": ["Biscoe", "Dream", "Torgersen"],
        "region": ["West", "West", "East"],
    }
)
print(island_region)

merged = penguins.merge(island_region, on="island", how="left")
print("\nafter merge, columns:", merged.shape[1])
print(merged["region"].value_counts().to_dict())
print("\nmean body mass (g) by region:")
print(merged.groupby("region")["body_mass_g"].mean().round(1))

A `left` merge keeps every penguin even if the lookup were missing, filling the new column with
`NaN`. Always check the row count after a merge: an accidental many-to-many key silently
duplicates rows.

In [ ]:
print("rows before merge:", len(penguins))
print("rows after merge :", len(merged))
print("duplicated rows  :", len(merged) - len(penguins))

### Reshaping with pivot_table

A pivot turns a long group-by into a grid. `mean` is the default aggregator; here we ask for it
explicitly, with species down the side and islands across the top.

In [ ]:
grid = penguins.pivot_table(
    index="species",
    columns="island",
    values="body_mass_g",
    aggfunc="mean",
)
grid.round(0)

### A quick look

Numbers are the point, but a picture catches surprises. The packaged helpers return `(fig, ax)`.

In [ ]:
fig, ax = histogram(penguins["body_mass_g"].dropna(), bins=25,
                    title="Body mass", xlabel="Body mass (g)")

clean = penguins.dropna(subset=["flipper_length_mm", "body_mass_g"])
fig, ax = scatterplot(clean["flipper_length_mm"], clean["body_mass_g"],
                      title="Flipper length vs body mass",
                      xlabel="Flipper length (mm)", ylabel="Body mass (g)")

The scatter shows a clear positive relationship with a gap between the long-flippered Gentoo and
the other two species — the kind of structure a group-by summary alone can hide.

## Exercises

1. **Standardise by group.** For each species, compute the z-score of `body_mass_g` within that
   species using `groupby` and `transform`. Add it as a column and report each species' mean
   z-score (it should be ~0).
2. **Mask and count.** Using a NumPy boolean mask on the flipper-length column, count how many
   penguins have flippers longer than 220 mm, and what share of the clean rows that is.
3. **Merge and pivot.** Build a lookup giving each island its region, merge it in, and produce a
   pivot table of mean `body_mass_g` with species as rows and region as columns.

## Limitations

Everything here fits in memory. For tables larger than RAM you need chunked reads, a database, or
a different engine such as Polars or DuckDB. pandas infers dtypes, and that inference can be
wrong for identifiers, so check `info()` rather than assuming. A `merge` can quietly multiply
rows if the key is not unique, which is why the row-count check above matters. Finally,
`groupby` drops missing values by default, so a group's mean is computed only over the rows that
have a value — a choice worth stating whenever you report a number.